# BKK 3-Class V3 Recall-Tuned Exact-Hour Rain Intensity Models

This notebook uses the same two-stage setup as `BKK_3_Class_V3`, but changes threshold tuning to focus on catching more `moderate_or_heavy` rain.

Final classes:

- `0 = no_rain`: `< 0.1 mm`
- `1 = light`: `0.1 mm` to `< 2.5 mm`
- `2 = moderate_or_heavy`: `>= 2.5 mm`

The V3 threshold search optimizes the `moderate_or_heavy` F2 score on validation data. F2 weights recall more than precision, so this version should catch more stronger-rain events, at the cost of more false alarms.


## 1. Setup


In [ ]:
import json
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg2
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)

pd.set_option("display.max_columns", 180)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)


## 2. Configuration


In [ ]:
DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE", "postgres"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "Pass1234"),
}

TABLE_NAME = '"OM_BKK_DATA"'
PRECOMPUTE_TABLE_NAME = '"OM_BKK_DATA_PRECOMPUTE"'
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / "ML_Model_V2" / "trained_models" / "BKK_3_Class_V3"

HORIZONS = [1, 2, 3, 4, 5, 6]
CLASS_LABELS = {
    0: "no_rain",
    1: "light",
    2: "moderate_or_heavy",
}
CLASS_ORDER = list(CLASS_LABELS)
INTENSITY_THRESHOLDS_MM = {
    "rain_min": 0.1,
    "moderate_or_heavy_min": 2.5,
}

RAIN_PROBABILITY_THRESHOLDS_TO_TEST = np.arange(0.20, 0.81, 0.05)
STRONG_PROBABILITY_THRESHOLDS_TO_TEST = np.arange(0.01, 0.76, 0.02)
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
STRONG_RECALL_BETA = 2.0
SAMPLE_ROWS = None  # Example for quick smoke tests: 300_000
RANDOM_STATE = 42


## 3. Feature Columns


In [ ]:
BASELINE_FEATURE_COLUMNS = [
    "temperature_2m", "relative_humidity_2m", "pressure_msl", "surface_pressure",
    "dew_point_2m", "precipitation", "cloud_cover", "wind_speed_10m",
    "wind_direction_10m", "temperature_dew_point_spread", "pressure_msl_change_3h",
    "pressure_msl_change_6h", "precipitation_lag_1h", "precipitation_lag_2h",
    "precipitation_lag_3h", "precipitation_lag_6h", "precipitation_sum_past_3h",
    "precipitation_sum_past_6h", "precipitation_sum_past_12h", "precipitation_sum_past_24h",
    "cloud_cover_lag_1h", "cloud_cover_lag_3h", "cloud_cover_lag_6h",
    "humidity_lag_1h", "humidity_lag_3h", "humidity_lag_6h",
    "wind_speed_lag_1h", "wind_speed_lag_3h", "hour_sin", "hour_cos",
    "month_sin", "month_cos", "grid_row", "grid_column", "latitude", "longitude",
]

NEIGHBOR_FEATURE_COLUMNS = [
    "neighbor_count", "neighbor_precipitation_mean", "neighbor_precipitation_max",
    "neighbor_precipitation_sum", "neighbor_rain_count", "neighbor_rain_rate",
    "neighbor_cloud_cover_mean", "neighbor_cloud_cover_max", "neighbor_relative_humidity_mean",
    "neighbor_relative_humidity_max", "neighbor_pressure_msl_mean", "neighbor_pressure_msl_min",
    "neighbor_pressure_msl_max", "neighbor_temperature_2m_mean", "neighbor_dew_point_2m_mean",
    "neighbor_temperature_dew_point_spread_mean", "neighbor_wind_speed_10m_mean",
    "neighbor_wind_speed_10m_max", "row_minus_precipitation_mean", "row_plus_precipitation_mean",
    "column_minus_precipitation_mean", "column_plus_precipitation_mean", "row_minus_cloud_cover_mean",
    "row_plus_cloud_cover_mean", "column_minus_cloud_cover_mean", "column_plus_cloud_cover_mean",
    "neighbor_precipitation_mean_minus_center", "neighbor_cloud_cover_mean_minus_center",
    "neighbor_relative_humidity_mean_minus_center", "center_pressure_msl_minus_neighbor_mean",
]

FEATURE_COLUMNS = BASELINE_FEATURE_COLUMNS + NEIGHBOR_FEATURE_COLUMNS
FUTURE_PRECIP_COLUMNS = [f"precipitation_next_{horizon}h" for horizon in HORIZONS]
FINAL_TARGET_COLUMNS = [f"rain_3class_next_{horizon}h" for horizon in HORIZONS]
RAIN_TARGET_COLUMNS = [f"rain_binary_next_{horizon}h" for horizon in HORIZONS]
STRONG_TARGET_COLUMNS = [f"strong_if_rain_next_{horizon}h" for horizon in HORIZONS]
TARGET_COLUMNS = FINAL_TARGET_COLUMNS + RAIN_TARGET_COLUMNS + STRONG_TARGET_COLUMNS

print(f"Features used: {len(FEATURE_COLUMNS)}")
print(f"Final 3-class targets: {FINAL_TARGET_COLUMNS}")
print(f"Class labels: {CLASS_LABELS}")


## 4. Load Precomputed Features And Build Targets


In [ ]:
def connect():
    return psycopg2.connect(**DB_CONFIG)


def rain_3class(precipitation_mm):
    bins = [
        -np.inf,
        INTENSITY_THRESHOLDS_MM["rain_min"],
        INTENSITY_THRESHOLDS_MM["moderate_or_heavy_min"],
        np.inf,
    ]
    return pd.cut(
        precipitation_mm,
        bins=bins,
        labels=CLASS_ORDER,
        right=False,
    ).astype("int8")


def read_training_data(sample_rows=None):
    select_columns = [
        "grid_number",
        "local_forecast_time AS forecast_time",
        *FEATURE_COLUMNS,
        *FUTURE_PRECIP_COLUMNS,
    ]
    select_sql = ",\n        ".join(select_columns)
    query = f'''
    SELECT
        {select_sql}
    FROM {PRECOMPUTE_TABLE_NAME}
    WHERE pressure_msl_change_6h IS NOT NULL
      AND precipitation_lag_6h IS NOT NULL
      AND precipitation_sum_past_24h IS NOT NULL
      AND cloud_cover_lag_6h IS NOT NULL
      AND humidity_lag_6h IS NOT NULL
      AND wind_speed_lag_3h IS NOT NULL
      AND precipitation_next_{max(HORIZONS)}h IS NOT NULL
      AND neighbor_count > 0
    ORDER BY local_forecast_time, grid_number
    '''
    if sample_rows:
        query = f'''
        SELECT *
        FROM ({query}) complete_rows
        ORDER BY random()
        LIMIT {int(sample_rows)}
        '''
    with connect() as conn:
        data = pd.read_sql_query(query, conn, parse_dates=["forecast_time"])

    for horizon in HORIZONS:
        final_target = f"rain_3class_next_{horizon}h"
        rain_target = f"rain_binary_next_{horizon}h"
        strong_target = f"strong_if_rain_next_{horizon}h"
        data[final_target] = rain_3class(data[f"precipitation_next_{horizon}h"])
        data[rain_target] = (data[final_target] > 0).astype("int8")
        data[strong_target] = (data[final_target] == 2).astype("int8")
    return data


In [ ]:
df = read_training_data(SAMPLE_ROWS)
print(df.shape)
print(df["forecast_time"].min(), "to", df["forecast_time"].max())
display(df.head())

required_columns = sorted(set(FEATURE_COLUMNS + FUTURE_PRECIP_COLUMNS + TARGET_COLUMNS))
missing_counts = df[required_columns].isna().sum().sort_values(ascending=False)
display(missing_counts[missing_counts > 0])


## 5. Class Balance


In [ ]:
balance_rows = []
for horizon in HORIZONS:
    target = f"rain_3class_next_{horizon}h"
    counts = df[target].value_counts().reindex(CLASS_ORDER, fill_value=0)
    for class_id, rows in counts.items():
        balance_rows.append({
            "horizon_h": horizon,
            "target_column": target,
            "class_id": int(class_id),
            "class_label": CLASS_LABELS[int(class_id)],
            "rows": int(rows),
            "class_rate": float(rows / len(df)),
        })

target_balance = pd.DataFrame(balance_rows)
display(target_balance)

sns.barplot(data=target_balance, x="horizon_h", y="class_rate", hue="class_label")
plt.title("BKK 3-Class V3 Class Rate By Horizon")
plt.xlabel("Forecast horizon: exact hour t+h")
plt.ylabel("Class rate")
plt.show()


## 6. Chronological Split


In [ ]:
def add_time_split(data, train_fraction=0.70, validation_fraction=0.15):
    unique_times = np.array(sorted(data["forecast_time"].unique()))
    train_end = unique_times[int(len(unique_times) * train_fraction)]
    validation_end = unique_times[int(len(unique_times) * (train_fraction + validation_fraction))]
    out = data.copy()
    out["split"] = "test"
    out.loc[out["forecast_time"] < train_end, "split"] = "train"
    out.loc[(out["forecast_time"] >= train_end) & (out["forecast_time"] < validation_end), "split"] = "validation"
    return out, train_end, validation_end

model_df, train_end, validation_end = add_time_split(df, TRAIN_FRACTION, VALIDATION_FRACTION)
print(f"Train before: {train_end}")
print(f"Validation before: {validation_end}")
display(model_df.groupby("split").size().rename("rows").reset_index())

train_df = model_df[model_df["split"] == "train"]
validation_df = model_df[model_df["split"] == "validation"]
test_df = model_df[model_df["split"] == "test"]

x_train = train_df[FEATURE_COLUMNS].astype("float32")
x_validation = validation_df[FEATURE_COLUMNS].astype("float32")
x_test = test_df[FEATURE_COLUMNS].astype("float32")


## 7. Two-Stage Evaluation Helpers


In [ ]:
def predict_three_class(rain_probability, strong_probability, rain_threshold, strong_threshold):
    predictions = np.zeros(len(rain_probability), dtype="int8")
    rain_mask = rain_probability >= rain_threshold
    predictions[rain_mask] = 1
    predictions[rain_mask & (strong_probability >= strong_threshold)] = 2
    return predictions


def class_metric_rows(y_true, y_pred, rain_probability, strong_probability, model_name, horizon, split, rain_threshold, strong_threshold):
    rows = []
    for class_id, class_label in CLASS_LABELS.items():
        binary_true = (y_true == class_id).astype("int8")
        binary_pred = (y_pred == class_id).astype("int8")
        rows.append({
            "model": model_name,
            "target_type": "exact_hour_3class_two_stage_recall_tuned",
            "feature_set": "neighbor_grid",
            "horizon_h": horizon,
            "split": split,
            "class_id": class_id,
            "class_label": class_label,
            "rows": int(len(y_true)),
            "class_rate": float(binary_true.mean()),
            "predicted_class_rate": float(binary_pred.mean()),
            "precision": float(precision_score(binary_true, binary_pred, zero_division=0)),
            "recall": float(recall_score(binary_true, binary_pred, zero_division=0)),
            "f1": float(f1_score(binary_true, binary_pred, zero_division=0)),
            "rain_threshold": float(rain_threshold),
            "strong_threshold": float(strong_threshold),
            "mean_rain_probability": float(rain_probability.mean()),
            "mean_strong_probability": float(strong_probability.mean()),
        })
    return rows


def evaluate_predictions(y_true, y_pred, rain_probability, strong_probability, model_name, horizon, split, rain_threshold, strong_threshold):
    base = {
        "model": model_name,
        "target_type": "exact_hour_3class_two_stage_recall_tuned",
        "feature_set": "neighbor_grid",
        "horizon_h": horizon,
        "split": split,
        "rows": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=CLASS_ORDER, average="weighted", zero_division=0)),
        "rain_threshold": float(rain_threshold),
        "strong_threshold": float(strong_threshold),
    }
    class_rows = class_metric_rows(
        y_true, y_pred, rain_probability, strong_probability, model_name, horizon, split, rain_threshold, strong_threshold
    )
    confusion = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=CLASS_ORDER),
        index=[f"actual_{CLASS_LABELS[i]}" for i in CLASS_ORDER],
        columns=[f"predicted_{CLASS_LABELS[i]}" for i in CLASS_ORDER],
    )
    confusion.insert(0, "horizon_h", horizon)
    confusion.insert(1, "split", split)
    confusion.insert(2, "rain_threshold", float(rain_threshold))
    confusion.insert(3, "strong_threshold", float(strong_threshold))
    return base, class_rows, confusion


def binary_fbeta(precision, recall, beta=STRONG_RECALL_BETA):
    if precision + recall == 0:
        return 0.0
    beta_sq = beta ** 2
    return float((1 + beta_sq) * precision * recall / ((beta_sq * precision) + recall))


def tune_thresholds(y_true, rain_probability, strong_probability):
    rows = []
    best = None
    for rain_threshold in RAIN_PROBABILITY_THRESHOLDS_TO_TEST:
        for strong_threshold in STRONG_PROBABILITY_THRESHOLDS_TO_TEST:
            y_pred = predict_three_class(rain_probability, strong_probability, rain_threshold, strong_threshold)
            macro_f1 = f1_score(y_true, y_pred, labels=CLASS_ORDER, average="macro", zero_division=0)
            strong_true = (y_true == 2).astype("int8")
            strong_pred = (y_pred == 2).astype("int8")
            strong_recall = recall_score(strong_true, strong_pred, zero_division=0)
            strong_precision = precision_score(strong_true, strong_pred, zero_division=0)
            strong_f1 = f1_score(strong_true, strong_pred, zero_division=0)
            strong_f2 = binary_fbeta(strong_precision, strong_recall)
            rows.append({
                "rain_threshold": float(rain_threshold),
                "strong_threshold": float(strong_threshold),
                "macro_f1": float(macro_f1),
                "moderate_or_heavy_precision": float(strong_precision),
                "moderate_or_heavy_recall": float(strong_recall),
                "moderate_or_heavy_f1": float(strong_f1),
                "moderate_or_heavy_f2": float(strong_f2),
            })
            candidate = rows[-1]
            if best is None:
                best = candidate
            else:
                candidate_key = (
                    candidate["moderate_or_heavy_f2"],
                    candidate["moderate_or_heavy_recall"],
                    candidate["macro_f1"],
                    candidate["moderate_or_heavy_precision"],
                )
                best_key = (
                    best["moderate_or_heavy_f2"],
                    best["moderate_or_heavy_recall"],
                    best["macro_f1"],
                    best["moderate_or_heavy_precision"],
                )
                if candidate_key > best_key:
                    best = candidate
    return best, pd.DataFrame(rows)


## 8. Train Two-Stage Models


In [ ]:
try:
    from lightgbm import LGBMClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install LightGBM first: %pip install lightgbm") from exc

rain_models = {}
strong_models = {}
threshold_rows = []
threshold_search_tables = []
metric_rows = []
class_metric_rows_all = []
confusion_tables = []
feature_importance_rows = []

for horizon in HORIZONS:
    final_target = f"rain_3class_next_{horizon}h"
    rain_target = f"rain_binary_next_{horizon}h"
    strong_target = f"strong_if_rain_next_{horizon}h"
    print(f"Training two-stage models for exact +{horizon}h...")

    y_rain_train = train_df[rain_target].astype("int8")
    y_rain_validation = validation_df[rain_target].astype("int8")
    y_final_validation = validation_df[final_target].astype("int8").to_numpy()
    y_final_test = test_df[final_target].astype("int8").to_numpy()

    rain_model = LGBMClassifier(
        objective="binary",
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    rain_model.fit(
        x_train,
        y_rain_train,
        eval_set=[(x_validation, y_rain_validation)],
        eval_metric="binary_logloss",
    )
    rain_models[horizon] = rain_model

    rainy_train = train_df[rain_target] == 1
    rainy_validation = validation_df[rain_target] == 1
    if rainy_train.sum() == 0:
        raise RuntimeError(f"No rainy training rows for horizon {horizon}.")

    strong_model = LGBMClassifier(
        objective="binary",
        class_weight="balanced",
        n_estimators=500,
        learning_rate=0.04,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    strong_model.fit(
        train_df.loc[rainy_train, FEATURE_COLUMNS].astype("float32"),
        train_df.loc[rainy_train, strong_target].astype("int8"),
        eval_set=[(
            validation_df.loc[rainy_validation, FEATURE_COLUMNS].astype("float32"),
            validation_df.loc[rainy_validation, strong_target].astype("int8"),
        )],
        eval_metric="binary_logloss",
    )
    strong_models[horizon] = strong_model

    rain_validation_probability = rain_model.predict_proba(x_validation)[:, 1]
    strong_validation_probability = strong_model.predict_proba(x_validation)[:, 1]
    best_thresholds, threshold_search = tune_thresholds(
        y_final_validation, rain_validation_probability, strong_validation_probability
    )
    best_thresholds["horizon_h"] = horizon
    threshold_rows.append(best_thresholds)
    threshold_search["horizon_h"] = horizon
    threshold_search_tables.append(threshold_search)

    for model_stage, model in [("rain_binary", rain_model), ("strong_if_rain", strong_model)]:
        importances = np.asarray(model.feature_importances_)
        if importances.ndim == 1:
            importances_by_feature = importances
        else:
            importances_by_feature = importances.sum(axis=1)
        for feature, importance in zip(FEATURE_COLUMNS, importances_by_feature):
            feature_importance_rows.append({
                "model": "lightgbm",
                "model_stage": model_stage,
                "horizon_h": horizon,
                "feature": feature,
                "importance": float(importance),
                "is_neighbor_feature": feature in NEIGHBOR_FEATURE_COLUMNS,
            })

    for split_name, x_split, split_df, y_final in [
        ("validation", x_validation, validation_df, y_final_validation),
        ("test", x_test, test_df, y_final_test),
    ]:
        rain_probability = rain_model.predict_proba(x_split)[:, 1]
        strong_probability = strong_model.predict_proba(x_split)[:, 1]
        y_pred = predict_three_class(
            rain_probability,
            strong_probability,
            best_thresholds["rain_threshold"],
            best_thresholds["strong_threshold"],
        )
        metrics, class_rows, confusion = evaluate_predictions(
            y_final,
            y_pred,
            rain_probability,
            strong_probability,
            "lightgbm_two_stage",
            horizon,
            split_name,
            best_thresholds["rain_threshold"],
            best_thresholds["strong_threshold"],
        )
        metric_rows.append(metrics)
        class_metric_rows_all.extend(class_rows)
        confusion_tables.append(confusion)

thresholds = pd.DataFrame(threshold_rows)
threshold_search_results = pd.concat(threshold_search_tables, ignore_index=True)
all_metrics = pd.DataFrame(metric_rows)
class_metrics = pd.DataFrame(class_metric_rows_all)
confusion_results = pd.concat(confusion_tables, ignore_index=True)
feature_importance = pd.DataFrame(feature_importance_rows)

display(thresholds.sort_values("horizon_h"))
display(all_metrics.sort_values(["horizon_h", "split"]))
display(class_metrics[class_metrics["split"] == "test"].sort_values(["horizon_h", "class_id"]))
display(confusion_results[confusion_results["split"] == "test"])


## 9. Results Charts


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)
sns.lineplot(data=all_metrics[all_metrics["split"] == "test"], x="horizon_h", y="macro_f1", marker="o", ax=axes[0])
axes[0].set_title("BKK 3-Class V3 Test Macro F1 By Horizon")
sns.lineplot(data=all_metrics[all_metrics["split"] == "test"], x="horizon_h", y="balanced_accuracy", marker="o", ax=axes[1])
axes[1].set_title("BKK 3-Class V3 Test Balanced Accuracy By Horizon")
for ax in axes:
    ax.set_xlabel("Forecast horizon: exact hour t+h")
plt.show()

sns.catplot(
    data=class_metrics[class_metrics["split"] == "test"],
    x="horizon_h",
    y="f1",
    hue="class_label",
    kind="bar",
    height=5,
    aspect=1.8,
)
plt.title("BKK 3-Class V3 Test F1 By Class")
plt.xlabel("Forecast horizon: exact hour t+h")
plt.ylabel("F1")
plt.show()

display(feature_importance.sort_values(["model_stage", "horizon_h", "importance"], ascending=[True, True, False]).groupby(["model_stage", "horizon_h"]).head(10))


## 10. Save BKK 3-Class V3 Models And Results


In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for horizon, model in rain_models.items():
    path = MODEL_DIR / f"bkk_3_class_v3_exact_next_{horizon}h_rain_binary_lightgbm.joblib"
    joblib.dump(model, path)

for horizon, model in strong_models.items():
    path = MODEL_DIR / f"bkk_3_class_v3_exact_next_{horizon}h_strong_if_rain_lightgbm.joblib"
    joblib.dump(model, path)

target_balance.to_csv(MODEL_DIR / "bkk_3_class_v3_target_balance.csv", index=False)
thresholds.to_csv(MODEL_DIR / "bkk_3_class_v3_thresholds.csv", index=False)
threshold_search_results.to_csv(MODEL_DIR / "bkk_3_class_v3_threshold_search.csv", index=False)
all_metrics.to_csv(MODEL_DIR / "bkk_3_class_v3_metrics.csv", index=False)
class_metrics.to_csv(MODEL_DIR / "bkk_3_class_v3_class_metrics.csv", index=False)
confusion_results.to_csv(MODEL_DIR / "bkk_3_class_v3_confusion_matrices.csv", index=False)
feature_importance.to_csv(MODEL_DIR / "bkk_3_class_v3_lightgbm_feature_importance.csv", index=False)

metadata = {
    "table": TABLE_NAME,
    "precompute_table": PRECOMPUTE_TABLE_NAME,
    "target_type": "rain_3class_exact_at_t_plus_h_two_stage_recall_tuned",
    "horizons": HORIZONS,
    "intensity_thresholds_mm": INTENSITY_THRESHOLDS_MM,
    "class_labels": CLASS_LABELS,
    "model": "two_stage_lightgbm",
    "stage_1": "rain_binary",
    "stage_2": "strong_if_rain",
    "feature_set": "neighbor_grid",
    "feature_columns": FEATURE_COLUMNS,
    "future_precip_columns": FUTURE_PRECIP_COLUMNS,
    "final_target_columns": FINAL_TARGET_COLUMNS,
    "rain_target_columns": RAIN_TARGET_COLUMNS,
    "strong_target_columns": STRONG_TARGET_COLUMNS,
    "train_end_exclusive": str(train_end),
    "validation_end_exclusive": str(validation_end),
    "sample_rows": SAMPLE_ROWS,
    "model_dir": str(MODEL_DIR),
    "thresholds": thresholds.to_dict(orient="records"),
    "target_balance": target_balance.to_dict(orient="records"),
    "metrics": all_metrics.to_dict(orient="records"),
    "class_metrics": class_metrics.to_dict(orient="records"),
}
(MODEL_DIR / "bkk_3_class_v3_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved BKK 3-Class V3 models and result CSVs to {MODEL_DIR}")
